In [ ]:
import pandas as pd

In [ ]:
analyse_file = "analyses_2022.csv.zst"

analyse_df_sample = pd.read_csv(analyse_file, sep=";", nrows=1)

categorical_values = ("Lb", "Nom", "Mnemo", "Cd", "Sym", "Com")
categories = analyse_df_sample.columns[analyse_df_sample.columns.str.startswith(categorical_values)]

dict_categories = {col: "category" for col in categories}
dict_converters = {col: str for col in categories}
chunksize = 500_000

iterator = pd.read_csv(analyse_file, 
                       sep=";", 
                       na_values="",
                       escapechar="\\",
                       converters=dict_converters,
                       dtype = {'RsAna':'str'},
                       parse_dates=['DatePrel'],
                       chunksize=chunksize)
iterator

analyse_df = None
i=0
for df in iterator:
    print(i)
    i+=chunksize
    df = df.astype(dtype=dict_categories)
    #on enlève la teinte de l'eau (1739) et l'odeur (1416) qui donnent un résultat en chaine de caractère
    cond_remove_teinte = df.CdParametre!='1739'
    cond_remove_odeur = df.CdParametre!='1416'
    df = df[cond_remove_teinte & cond_remove_odeur].astype({'RsAna':'float64'})
    if analyse_df is None:
        analyse_df = df
    else:
        for col in dict_categories:
            cat_union = analyse_df[col].cat.categories.union(df[col].cat.categories);
            analyse_df[col]=analyse_df[col].cat.set_categories(cat_union)
            df[col]=df[col].cat.set_categories(cat_union)
        analyse_df= pd.concat([analyse_df,df],ignore_index=True)

analyse_df

In [ ]:
seq_eau = pd.read_csv("classe_alteration_seq_eau.csv",
                      na_values="\\N",
                      true_values=['t'],
                      false_values=['f'],
                      dtype={"parametre_id" :"str",'nom':'category','nom.1':'category','libelle':'category'}).astype({'parametre_id':'category'})
seq_eau.info()

In [ ]:
analyse_merged = analyse_df.merge(seq_eau,left_on="CdParametre",right_on="parametre_id")

cond_inf = analyse_merged.borne_inf.isna() | (analyse_merged.borne_inf_incluse & (analyse_merged.borne_inf <=  analyse_merged.RsAna)) | (analyse_merged.borne_inf < analyse_merged.RsAna)
cond_sup = analyse_merged.borne_sup.isna() | (analyse_merged.borne_sup_incluse & (analyse_merged.borne_sup >=  analyse_merged.RsAna)) | (analyse_merged.borne_sup > analyse_merged.RsAna)
analyse_merged = analyse_merged[cond_inf & cond_sup]

In [ ]:
analyse_merged.loc[:,['RsAna','CdParametre','parametre_id','borne_inf','borne_sup']]

In [ ]:
import pandas as pd
import geopandas as gpd
import shapely.geometry as geom
import os



df_stations = pd.read_csv("stations.csv.zst",sep=';',escapechar = '\\')


In [ ]:
crs_lambert = 'PROJCS["RGF_1993_Lambert_93",GEOGCS["GCS_RGF_1993",DATUM["D_RGF_1993",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic"],PARAMETER["False_Easting",700000.0],PARAMETER["False_Northing",6600000.0],PARAMETER["Central_Meridian",3.0],PARAMETER["Standard_Parallel_1",49.0],PARAMETER["Standard_Parallel_2",44.0],PARAMETER["Latitude_Of_Origin",46.5],UNIT["Meter",1.0]]'
x_col = 'CoordXStationMesureEauxSurface'
y_col = 'CoordYStationMesureEauxSurface'

carto = gpd.GeoDataFrame(df_stations,crs=crs_lambert ,
                        geometry = gpd.GeoSeries(df_stations.agg(lambda x:geom.Point(x.loc[x_col],x.loc[y_col])  ,axis=1)))

In [ ]:
carto